# Isolation Forest Training for Vibration Anomaly Detection

This notebook trains an Isolation Forest model on baseline vibration features and exports it to ONNX format.

**Task C5** — Phase 1C: ML Models

## Features (11 inputs)
1. `accel_rms_mean` - mean vibration RMS over 10min window
2. `accel_rms_std` - std deviation of vibration RMS
3. `accel_delta` - current - rolling mean
4. `accel_roc` - rate of change of vibration
5. `velocity_rms_z` - Z-score of velocity
6. `peak_to_rms_ratio` - peak/RMS ratio
7. `motor_temp_delta` - motor temp - environment temp
8. `humidity_trend` - slope of humidity over 30min
9. `load_pct` - load / max capacity
10. `load_variance` - std of load over 5min
11. `multivariate_score` - composite score (0-1)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, silhouette_score
import onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort

print("Libraries loaded successfully")

## 1. Load Sample Data

In [ ]:
# Load sample data
data_path = "data/sample_vibration_features.csv"
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} samples with {len(df.columns)} features")
print("\nFirst 5 rows:")
print(df.head())
print("\nFeature statistics:")
print(df.describe())

In [ ]:
# Visualize feature distributions
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.ravel()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=50, alpha=0.7)
    axes[i].set_title(col)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

## 2. Train Isolation Forest with Different Contamination Parameters

In [ ]:
# Prepare data
X = df.values
feature_names = list(df.columns)
print(f"Feature names: {feature_names}")
print(f"Data shape: {X.shape}")

In [ ]:
# Train Isolation Forest with different contamination parameters
contamination_values = [0.01, 0.05, 0.1, 0.15]
models = {}
results = {}

for cont in contamination_values:
    print(f"\nTraining with contamination={cont}...")
    
    model = IsolationForest(
        contamination=cont,
        random_state=42,
        n_estimators=100
    )
    model.fit(X)
    
    # Get anomaly scores (decision_function)
    scores = model.decision_function(X)
    predictions = model.predict(X)  # 1 for inlier, -1 for outlier
    
    # Calculate silhouette score
    sil_score = silhouette_score(X, predictions)
    
    # Count anomalies
    n_anomalies = (predictions == -1).sum()
    anomaly_pct = 100 * n_anomalies / len(predictions)
    
    models[cont] = model
    results[cont] = {
        'silhouette': sil_score,
        'n_anomalies': n_anomalies,
        'anomaly_pct': anomaly_pct,
        'score_mean': scores.mean(),
        'score_std': scores.std()
    }
    
    print(f"  Silhouette score: {sil_score:.4f}")
    print(f"  Anomalies detected: {n_anomalies} ({anomaly_pct:.2f}%)")
    print(f"  Score mean: {scores.mean():.4f}, std: {scores.std():.4f}")

In [ ]:
# Compare results
print("\n" + "="*60)
print("Comparison of contamination parameters:")
print("="*60)
print(f"{'Contamination':<15} {'Silhouette':<12} {'Anomalies':<12} {'%':<10}")
print("-"*60)

for cont in contamination_values:
    r = results[cont]
    print(f"{cont:<15} {r['silhouette']:<12.4f} {r['n_anomalies']:<12} {r['anomaly_pct']:<10.2f}")

In [ ]:
# Visualize anomaly score distributions for each contamination value
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, cont in enumerate(contamination_values):
    model = models[cont]
    scores = model.decision_function(X)
    
    axes[i].hist(scores, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    axes[i].axvline(x=0, color='red', linestyle='--', label='Decision boundary')
    axes[i].set_title(f'Contamination={cont} (detected {results[cont]["n_anomalies"]} anomalies)')
    axes[i].set_xlabel('Anomaly Score (decision_function)')
    axes[i].set_ylabel('Frequency')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Select Best Model and Train Final Version

In [ ]:
# Select best contamination based on silhouette score
best_cont = max(results, key=lambda c: results[c]['silhouette'])
print(f"Best contamination: {best_cont} (silhouette: {results[best_cont]['silhouette']:.4f})")

# Train final model with best parameter
final_model = models[best_cont]
print(f"\nFinal model: IsolationForest(contamination={best_cont})")
print(f"Number of estimators: {final_model.n_estimators}")
print(f"Number of features: {final_model.n_features_in_}")

## 4. Export to ONNX Format

In [ ]:
# Export model to ONNX
feature_names = list(df.columns)
n_features = len(feature_names)

# Define input type for ONNX
initial_type = [('float_input', FloatTensorType([None, n_features]))]

# Convert to ONNX
onnx_model = convert_sklearn(
    final_model,
    initial_types=initial_type,
    target_opset=15
)

print("Model converted to ONNX format")
print(f"ONNX model IR version: {onnx_model.ir_version}")

In [ ]:
# Save ONNX model
import os
os.makedirs("../models", exist_ok=True)

model_path = "../models/vibration_anomaly_v1.onnx"
with open(model_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"Model saved to: {model_path}")

# Verify the saved model
loaded_model = onnx.load(model_path)
print("\nVerifying saved model...")
print(f"Model inputs: {[inp.name for inp in loaded_model.graph.input]}")
print(f"Model outputs: {[out.name for out in loaded_model.graph.output]}")

## 5. Verify ONNX Model Produces Same Predictions

In [ ]:
# Load ONNX model with onnxruntime
session = ort.InferenceSession(model_path)
print(f"ONNX model inputs: {[inp.name for inp in session.get_inputs()]}")
print(f"ONNX model outputs: {[out.name for out in session.get_outputs()]}")

# Prepare test data (use first 100 samples)
test_data = X[:100].astype(np.float32)
input_name = session.get_inputs()[0].name

# Run inference with ONNX
onnx_output = session.run(None, {input_name: test_data})
print(f"\nONNX output shape: {[o.shape for o in onnx_output]}")

In [ ]:
# Compare sklearn predictions with ONNX predictions
# Note: skl2onnx exports decision_function output by default

# Get sklearn predictions
sklearn_scores = final_model.decision_function(test_data)

# Get ONNX predictions (first output is typically the decision_function)
onnx_scores = onnx_output[0].ravel()

print("Comparison of first 10 predictions:")
print(f"{'Index':<8} {'Sklearn':<15} {'ONNX':<15} {'Diff':<10}")
print("-"*50)
for i in range(10):
    print(f"{i:<8} {sklearn_scores[i]:<15.6f} {onnx_scores[i]:<15.6f} {abs(sklearn_scores[i]-onnx_scores[i]):<10.6f}")

# Check if predictions match
max_diff = np.abs(sklearn_scores - onnx_scores).max()
print(f"\nMax difference: {max_diff:.10f}")
print(f"Predictions match: {max_diff < 1e-5}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot of predictions
axes[0].scatter(sklearn_scores, onnx_scores, alpha=0.6)
axes[0].plot([sklearn_scores.min(), sklearn_scores.max()], [sklearn_scores.min(), sklearn_scores.max()], 'r--', label='y=x')
axes[0].set_xlabel('Sklearn decision_function')
axes[0].set_ylabel('ONNX decision_function')
axes[0].set_title('Sklearn vs ONNX Predictions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histogram of differences
diff = sklearn_scores - onnx_scores
axes[1].hist(diff, bins=50, alpha=0.7, color='green', edgecolor='black')
axes[1].axvline(x=0, color='red', linestyle='--', label='Zero')
axes[1].set_xlabel('Difference (Sklearn - ONNX)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Prediction Differences')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Summary and Next Steps

In [ ]:
print("="*60)
print("ISOLATION FOREST TRAINING COMPLETE")
print("="*60)
print(f"\nModel: IsolationForest")
print(f"Contamination: {best_cont}")
print(f"Number of features: {n_features}")
print(f"Feature names: {feature_names}")
print(f"\nONNX model saved to: {model_path}")
print(f"Sklearn and ONNX predictions match: {max_diff < 1e-5}")

print("\n--- Next Steps ---")
print("1. Use 'scripts/export_onnx.py' for CLI-based export")
print("2. Integrate with OnnxRuntime in the application")
print("3. Map anomaly scores to NORMAL/WARNING/CRITICAL status")
print("4. Test with real sensor data")
